# 02 — CLIP: Contrastive Vision-Language Pretraining

## What
CLIP (Contrastive Language-Image Pretraining, Radford et al. 2021) trains a visual encoder and a text encoder jointly to maximise the cosine similarity of matched image-text pairs and minimise it for unmatched pairs — using the InfoNCE (noise-contrastive estimation) loss over a batch.

## Why
CLIP enables zero-shot classification without task-specific training: encode all class labels as text, encode the query image, find the nearest label. It also produces rich visual features for downstream tasks. The 400M image-text pairs from the web provide the diversity that supervised ImageNet training lacks.

## Community context
CLIP spawned an ecosystem: ALIGN (Google, similar scale), OpenCLIP (open reproduction), SigLIP (sigmoid loss instead of softmax, scales better), and DFN (data filtering networks). CLIP embeddings are widely used in retrieval, generation (as CLIP-score), and zero-shot evaluation.

In [ ]:
# CLIP-style contrastive loss and zero-shot classification
import numpy as np

def normalize(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-8)

def clip_loss(image_features, text_features, temperature=0.07):
    """
    InfoNCE loss for a batch of (image, text) pairs.
    For each image, the matched text should be ranked highest.
    N = batch size.
    """
    img  = normalize(image_features)   # (N, D)
    txt  = normalize(text_features)     # (N, D)
    logits = img @ txt.T / temperature  # (N, N)
    labels = np.arange(len(img))        # diagonal is positive
    # Cross-entropy from image to text direction
    i2t = -np.mean([np.log(np.exp(logits[i, labels[i]]) / np.exp(logits[i]).sum())
                    for i in range(len(img))])
    # Cross-entropy from text to image direction
    t2i = -np.mean([np.log(np.exp(logits[labels[j], j]) / np.exp(logits[:, j]).sum())
                    for j in range(len(txt))])
    return (i2t + t2i) / 2

# Zero-shot classification
def zero_shot_classify(image_embed, class_text_embeds, class_names, temperature=0.07):
    img = normalize(image_embed[None])               # (1, D)
    txt = normalize(class_text_embeds)               # (C, D)
    logits = (img @ txt.T / temperature)[0]         # (C,)
    probs  = np.exp(logits) / np.exp(logits).sum()
    ranked = sorted(zip(class_names, probs), key=lambda x: -x[1])
    return ranked

np.random.seed(42)
D = 512  # embedding dim

# Simulate aligned embeddings: matched pairs are close, unmatched are far
N = 8
img_embeds  = np.random.randn(N, D)
txt_embeds  = img_embeds + np.random.randn(N, D) * 0.3  # matched: nearby

loss_matched   = clip_loss(img_embeds, txt_embeds)
# Shuffle text to create unmatched pairs
txt_shuffled = txt_embeds[np.random.permutation(N)]
loss_unmatched = clip_loss(img_embeds, txt_shuffled)
print(f'Loss (matched pairs):   {loss_matched:.4f}')
print(f'Loss (unmatched pairs): {loss_unmatched:.4f}')
print('Lower loss for matched pairs confirms InfoNCE is working\n')

# Zero-shot classification demo
classes = ['a photo of a cat', 'a photo of a dog', 'a photo of a car',
           'a photo of a bird', 'a photo of a building']
# Simulate: image is a cat, so cat text embedding is closest
cat_embed  = np.random.randn(D)
class_embeds = np.array([cat_embed + np.random.randn(D)*0.2,  # cat: very close
                         cat_embed + np.random.randn(D)*1.5,  # dog: further
                         np.random.randn(D),                  # car: random
                         np.random.randn(D),                  # bird: random
                         np.random.randn(D)])                 # building: random

results = zero_shot_classify(cat_embed, class_embeds, classes)
print('Zero-shot classification:')
for cls, prob in results:
    print(f'  {prob:.3f}  {cls}')

## CLIP-Score for Text-Image Alignment Evaluation

CLIP-Score measures how well a generated image matches its text prompt. It replaces expensive human evaluation for large-scale text-to-image model benchmarking (used in DALL-E, Stable Diffusion evaluations).

In [ ]:
# CLIP-Score computation
def clip_score(image_embed, text_embed, w=2.5):
    """
    CLIP-Score = w * max(cosine_similarity(I, T), 0)
    w=2.5 is the scale factor from Hessel et al. 2021.
    """
    cos = np.dot(normalize(image_embed), normalize(text_embed))
    return w * max(cos, 0)

np.random.seed(10)
base_txt = np.random.randn(D)
scenarios = [
    ('Perfect match',      base_txt + np.random.randn(D)*0.05),
    ('Good match',         base_txt + np.random.randn(D)*0.5),
    ('Partial match',      base_txt + np.random.randn(D)*1.5),
    ('Unrelated image',    np.random.randn(D)),
]
print('CLIP-Score (higher = better alignment):')
for name, img_e in scenarios:
    score = clip_score(img_e, base_txt)
    print(f'  {name:<20}: {score:.4f}')